# 대화형 승인·거부 — 수강생 실습본

원본: `src/raw/Chapter01/04-langgraph/05_hitl_interactive.py`  
대응 슬라이드: `docs/raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf`

**학습 목표:** 도구 앞 중단 → y/n/q 입력 → 상태 변경의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** LLM을 호출하고 터미널 입력을 기다린다. 승인하면 출력 모의 메일 도구가 실행되므로 호출 인자를 확인한다.



## 흐름을 먼저 읽기

도구 앞 중단 → y/n/q 입력 → 상태 변경

핵심은 State와 다음 노드의 결정 방식이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: 도구 앞 중단 → y/n/q 입력 → 상태 변경. 거부한 도구가 재실행되지 않도록 하는 코드는? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** LLM을 호출하고 터미널 입력을 기다린다. 승인하면 출력 모의 메일 도구가 실행되므로 호출 인자를 확인한다.



In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()


@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """이메일을 전송한다."""
    print(f'\n  [실제 전송] {recipient} / 제목: {subject}')
    return f'이메일이 {recipient}에게 성공적으로 전송되었습니다.'


tools = [send_email]
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    return {'messages': [llm_with_tools.invoke(state['messages'])]}


builder = StateGraph(State)
builder.add_node('agent', agent_node)
builder.add_node('tools', tool_node)
builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', tools_condition)
builder.add_edge('tools', 'agent')

app = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=['tools'],
)


def print_tool_calls(tool_calls: list[dict]) -> None:
    print('\n실행 대기 중인 도구 호출')
    for index, tool_call in enumerate(tool_calls, start=1):
        print(f'{index}. {tool_call["name"]}')
        for key, value in tool_call['args'].items():
            print(f'   - {key}: {value}')


def reject_tool_calls(config: dict, tool_calls: list[dict], reason: str) -> None:
    rejection_messages = [
        ToolMessage(
            content=f'사용자가 도구 실행을 거부했습니다. 사유: {reason}',
            tool_call_id=tool_call['id'],
        )
        for tool_call in tool_calls
    ]
    app.update_state(config, {'messages': rejection_messages}, as_node='tools')


def ask_human_decision() -> tuple[str, str]:
    print('\n도구를 실행할까요?')
    print('- y: 승인 후 실행')
    print('- n: 거부 후 에이전트에게 취소 사유 전달')
    print('- q: 프로그램 종료')

    decision = input('선택(y/n/q): ').strip().lower()
    if decision == 'n':
        reason = input('거부 사유를 입력하세요: ').strip()
        return decision, reason
    return decision, ''


def run_agent(user_input: str, thread_id: str = 'hitl_demo') -> None:
    config = {'configurable': {'thread_id': thread_id}}
    result = app.invoke({'messages': [HumanMessage(content=user_input)]}, config=config)

    while True:
        state = app.get_state(config)
        tool_calls = state.values['messages'][-1].tool_calls

        if not tool_calls:
            print('\n에이전트 응답')
            print(result['messages'][-1].content)
            break

        print_tool_calls(tool_calls)
        decision, reason = ask_human_decision()

        if decision == 'y':
            result = app.invoke(None, config=config)
        elif decision == 'n':
            reject_tool_calls(config, tool_calls, reason or '사유 없음')
            result = app.invoke(None, config=config)
        elif decision == 'q':
            print('프로그램을 종료합니다.')
            break
        else:
            print('y, n, q 중 하나를 입력하세요.')


if __name__ == '__main__':
    request = input('에이전트에게 요청할 내용을 입력하세요: ')
    run_agent(request)


## 결과를 검토하기

- 예상 동작: y/n/q 입력에 따라 승인·거부·종료 경로로 진행한다.
- 확인 질문: 거부한 도구가 재실행되지 않도록 하는 코드는?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
